# Weather Trend Forecasting: End-to-End Data Science Workflow

This notebook demonstrates every step of a typical machine learning workflow using the Global Weather Repository dataset.

## 1. Import Required Libraries
Import essential libraries for data analysis, visualization, and machine learning.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

## 2. Load and Explore the Dataset
Load the Global Weather Repository dataset and perform basic exploratory data analysis.

In [ ]:
# Load the dataset
weather_df = pd.read_csv('../data/Global Weather Repository.csv')

# Display the first few rows
weather_df.head()

In [ ]:
# Display dataset info and summary statistics
weather_df.info()
weather_df.describe().T

In [ ]:
# Check for missing values
weather_df.isnull().sum().sort_values(ascending=False).head(20)

## 3. Preprocess the Data
Handle missing values, encode categorical variables, and split the data into training and testing sets.

## 4. Exploratory Data Analysis (EDA) and Visualizations
Analyze trends, correlations, and patterns. Visualize temperature and precipitation.

In [ ]:
# --- EDA: Trends, Correlations, and Patterns ---
# 1. Correlation heatmap for numeric features
plt.figure(figsize=(14,10))
sns.heatmap(weather_df.corr(numeric_only=True), annot=False, cmap='coolwarm')
plt.title('Correlation Heatmap (Numeric Features)')
plt.show()

# 2. Distribution of temperature (if present)
if 'temperature' in weather_df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(weather_df['temperature'], kde=True, bins=50)
    plt.title('Temperature Distribution')
    plt.xlabel('Temperature')
    plt.show()

# 3. Distribution of precipitation (if present)
if 'precipitation' in weather_df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(weather_df['precipitation'], kde=True, bins=50)
    plt.title('Precipitation Distribution')
    plt.xlabel('Precipitation')
    plt.show()

# 4. Time series plot for temperature and precipitation (if date and columns exist)
if date_cols and 'temperature' in weather_df.columns:
    plt.figure(figsize=(14,5))
    plt.plot(weather_df[date_cols[0]], weather_df['temperature'], label='Temperature')
    if 'precipitation' in weather_df.columns:
        plt.plot(weather_df[date_cols[0]], weather_df['precipitation'], label='Precipitation')
    plt.legend()
    plt.title('Temperature and Precipitation Over Time')
    plt.xlabel('Date')
    plt.show()

## 5. Build and Evaluate Forecasting Models
Implement basic and advanced forecasting models. Evaluate with metrics.

In [ ]:
# --- Basic Forecasting Model: Decision Tree Regressor ---
from sklearn.tree import DecisionTreeRegressor

if X_train is not None:
    dt_model = DecisionTreeRegressor(random_state=42)
    dt_model.fit(X_train, y_train)
    dt_pred = dt_model.predict(X_test)
    dt_rmse = np.sqrt(mean_squared_error(y_test, dt_pred))
    dt_mae = mean_absolute_error(y_test, dt_pred)
    dt_r2 = r2_score(y_test, dt_pred)
    print(f"Decision Tree - RMSE: {dt_rmse:.2f}, MAE: {dt_mae:.2f}, R^2: {dt_r2:.2f}")
else:
    print("No target variable found for modeling.")

# --- Advanced Forecasting Model: Random Forest Regressor ---
from sklearn.ensemble import RandomForestRegressor

if X_train is not None:
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)
    rf_pred = rf_model.predict(X_test)
    rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
    rf_mae = mean_absolute_error(y_test, rf_pred)
    rf_r2 = r2_score(y_test, rf_pred)
    print(f"Random Forest - RMSE: {rf_rmse:.2f}, MAE: {rf_mae:.2f}, R^2: {rf_r2:.2f}")
else:
    print("No target variable found for modeling.")

# --- Model Comparison ---
if X_train is not None:
    results_df = pd.DataFrame({
        'Model': ['Decision Tree', 'Random Forest'],
        'RMSE': [dt_rmse, rf_rmse],
        'MAE': [dt_mae, rf_mae],
        'R2': [dt_r2, rf_r2]
    })
    display(results_df)


In [ ]:
# Example preprocessing: handle missing values, encode categoricals, scale features
# (Adjust columns as needed based on actual dataset)

# Impute missing numeric values with median
num_cols = weather_df.select_dtypes(include=[np.number]).columns
imputer = SimpleImputer(strategy='median')
weather_df[num_cols] = imputer.fit_transform(weather_df[num_cols])

# Encode categorical variables
cat_cols = weather_df.select_dtypes(include=['object']).columns
for col in cat_cols:
    if weather_df[col].nunique() < 20:
        weather_df[col] = LabelEncoder().fit_transform(weather_df[col].astype(str))

# Normalize numeric features
scaler = StandardScaler()
weather_df[num_cols] = scaler.fit_transform(weather_df[num_cols])

# Example: Split into train/test for a target variable (e.g., temperature)
target = 'temperature'  # Replace with actual target column name
if target in weather_df.columns:
    X = weather_df.drop(columns=[target])
    y = weather_df[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
else:
    X_train = X_test = y_train = y_test = None

In [ ]:
# --- Data Cleaning and Preprocessing ---
# 1. Parse date columns (if present)
date_cols = [col for col in weather_df.columns if 'date' in col.lower() or 'lastupdated' in col.lower()]
for col in date_cols:
    weather_df[col] = pd.to_datetime(weather_df[col], errors='coerce')

# 2. Handle missing values (already imputed numerics above)
# For categoricals with missing, fill with 'Unknown'
for col in weather_df.select_dtypes(include=['object']).columns:
    weather_df[col] = weather_df[col].fillna('Unknown')

# 3. Outlier detection and capping (example: z-score method for numerics)
from scipy.stats import zscore
z_thresh = 4
for col in weather_df.select_dtypes(include=[np.number]).columns:
    z = np.abs(zscore(weather_df[col], nan_policy='omit'))
    weather_df.loc[z > z_thresh, col] = np.sign(weather_df.loc[z > z_thresh, col]) * weather_df[col].mean()

# 4. Confirm cleaning
display(weather_df.info())
display(weather_df.describe().T)
display(weather_df.isnull().sum().sort_values(ascending=False).head(10))

## 4. Build and Train a Model
Select and train a machine learning model (e.g., Decision Tree Regressor) on the training data.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

if X_train is not None:
    model = DecisionTreeRegressor(random_state=42)
    model.fit(X_train, y_train)
else:
    model = None

## 5. Evaluate the Model
Assess the model's performance using metrics such as RMSE, MAE, and R².

In [ ]:
if model is not None:
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f'RMSE: {rmse:.2f}')
    print(f'MAE: {mae:.2f}')
    print(f'R^2: {r2:.2f}')

## 6. Make Predictions
Use the trained model to make predictions on new or test data and display the results.

In [ ]:
if model is not None:
    # Show first 10 predictions vs actual
    results = pd.DataFrame({'Actual': y_test[:10].values, 'Predicted': y_pred[:10]})
    display(results)
    # Plot actual vs predicted
    plt.figure(figsize=(8,5))
    plt.plot(y_test[:50].values, label='Actual')
    plt.plot(y_pred[:50], label='Predicted')
    plt.legend()
    plt.title('Actual vs Predicted (First 50 Samples)')
    plt.show()